<a href="https://colab.research.google.com/github/lisa11323/Organic-food/blob/All-data/%E1%84%8B%E1%85%B5%E1%86%AB%E1%84%89%E1%85%B3%E1%84%90%E1%85%A1_%E1%84%8F%E1%85%B3%E1%86%AF%E1%84%85%E1%85%A9%E1%84%85%E1%85%B5%E1%86%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup

In [ ]:
!pip install selenium

In [ ]:
url2="https://www.instagram.com/"

In [ ]:
pip install webdriver_manager

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
import time
import re
import unicodedata

In [ ]:
!pip install langdetect

In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

In [ ]:
driver.get(url2)
driver.implicitly_wait(3)

In [ ]:
def insta_search(word):

    url = 'https://www.instagram.com/explore/tags/' + word
    return url

In [ ]:
word = "beyondmeat"
url = insta_search(word)
driver.get(url)
url = insta_search(word)
driver.get(url)

In [ ]:
def post_first(driver):
    first = driver.find_element(By.CSS_SELECTOR, "div._aagw")

    first.click()
    time.sleep(3)

post_first(driver)

In [ ]:
from bs4 import BeautifulSoup
import re
import unicodedata
from langdetect import detect
from datetime import datetime

# 全局集合用于记录已处理的帖子内容
processed_posts = set()

def meat(driver):
    global processed_posts  # 使用全局变量记录已处理的帖子内容
    html = driver.page_source
    bs_obj = BeautifulSoup(html, "html.parser")

    # 提取内容
    try:
        content = bs_obj.select("div._a9zs>h1")[0].text
        content = unicodedata.normalize('NFC', content)
    except:
        content = ''

    if not content.strip():
        print("Empty or invalid post, skipped.")
        return None

    # 检查重复内容
    if content in processed_posts:
        print("Duplicate post detected, skipped.")
        return None

    # 添加到已处理集合
    processed_posts.add(content)

    try:
        if detect(content) != 'en':
            print("Non-English post detected, skipped.")
            return None
    except Exception as e:
        print("Error processing post:", e)
        return None

    # 提取标签
    tags = re.findall(r'#[^\s#,\\]+', content)

    # 提取日期
    try:
        date = bs_obj.select('time._a9ze._a9zf')[0]['datetime'][:10]
    except:
        date = ''

    # 日期筛选逻辑
    try:
        post_date = datetime.strptime(date, "%Y-%m-%d")
        target_date = datetime.strptime("2019-05-02", "%Y-%m-%d")
        if post_date < target_date:
            print(f"Post date {date} is before 2019-05-02, skipped.")
            return None
    except Exception as e:
        print(f"Error processing date: {e}")
        return None

    # 提取点赞数
    try:
        like = bs_obj.select('section.x12nagc.x182iqb8.x1pi30zi.x1swvt13 span')[2].text
    except:
        like = 0

    # 提取用户ID
    try:
        user_id = bs_obj.select('a._acan._acao._acat._acaw._aj1-._ap30._a6hd')[0].text.strip()
    except:
        user_id = ''

    # 返回数据
    data = [content, date, tags, like, user_id]
    return data


In [ ]:
meat(driver)

['Can you guess how this was done!? 🍔✨(sound on) 🔊 . If you support my work, like, leave a comment, and share with a friend!.Follow @vfx_monkey for more!....#magicofthemonth #vfxcompany #vfxanimation #vfxworld #vfxlife #vfxguru #vfxpro #vfxartists #vfxcompositing #vfxsupervisor #vfxartist #animationart #motiondevotion #3dartist #3danimation #motiondesigners #vfx #visualeffectsartist #bestvisualeffects #visual_creatorz #visualeffectsartists #visualeffectssupervisor #zachkingvideo #videowizard #cgiartist #zachkingmagic #escreators #zachking #beyondmeat #beyondmeatburger',
 '2020-11-20',
 ['#magicofthemonth',
  '#vfxcompany',
  '#vfxanimation',
  '#vfxworld',
  '#vfxlife',
  '#vfxguru',
  '#vfxpro',
  '#vfxartists',
  '#vfxcompositing',
  '#vfxsupervisor',
  '#vfxartist',
  '#animationart',
  '#motiondevotion',
  '#3dartist',
  '#3danimation',
  '#motiondesigners',
  '#vfx',
  '#visualeffectsartist',
  '#bestvisualeffects',
  '#visual_creatorz',
  '#visualeffectsartists',
  '#visualeffect

In [ ]:
def move_next(driver):

    right = driver.find_element(By.CSS_SELECTOR, "div._aaqg._aaqh button")
    right.click()
    time.sleep(3)

In [ ]:
results=[]
target=15000

for i in range(target):
    try:
        data=meat(driver)
        results.append(data)
        move_next(driver)
    except:
        time.sleep(3)
        move_next(driver)

print(results)

In [ ]:
import pandas as pd
filtered_results = [result for result in results if result is not None]

results_df = pd.DataFrame(filtered_results, columns=['content', 'date', 'tags', 'user_id'])

results_df.to_excel('beyongdmeat_123456.xlsx', index=False)